**CI twin of `ch09-knn.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.model_selection import train_test_split
import math

df = load_csv("penguins")
two = df[df["species"].isin(["Adelie", "Chinstrap"])].dropna(
    subset=["bill_length_mm", "body_mass_g"])
y = (two["species"] == "Chinstrap").astype(int)
X = two[["bill_length_mm", "body_mass_g"]]

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

probe = Xte.iloc[0]     # one held-out bird: bill 37.8 mm, mass 4250 g
dists = ((Xtr - probe) ** 2).sum(axis=1) ** 0.5
nearest5 = dists.nsmallest(5)

print(f"probe bird: bill {probe['bill_length_mm']} mm, "
      f"mass {probe['body_mass_g']:.0f} g")
print("its 5 nearest recorded birds:")
for idx, d in nearest5.items():
    label = "Chinstrap" if ytr.loc[idx] == 1 else "Adelie"
    print(f"  distance {d:6.1f}   bill {Xtr.loc[idx, 'bill_length_mm']:>5} mm  "
          f"mass {Xtr.loc[idx, 'body_mass_g']:6.0f} g   {label}")
votes = [ytr.loc[i] for i in nearest5.index]
print(f"vote: {sum(votes)} of 5 say Chinstrap -> predict "
      f"{'Chinstrap' if sum(votes) > 2 else 'Adelie'}")

In [ ]:
# Bird A: bill wildly different (+10 mm), mass nearly identical (+50 g)
dist_a = math.sqrt(10**2 + 50**2)
# Bird B: bill nearly identical (+1 mm), mass off by a routine 300 g
dist_b = math.sqrt(1**2 + 300**2)

print(f"bird A (bill +10 mm, mass  +50 g): distance {dist_a:.1f}")
print(f"bird B (bill  +1 mm, mass +300 g): distance {dist_b:.1f}")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

raw = KNeighborsClassifier(n_neighbors=5).fit(Xtr, ytr)
print(f"held-out accuracy, raw features: "
      f"{accuracy_score(yte, raw.predict(Xte)):.3f}")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(Xtr)          # learns mean/std from TRAIN only
Xtr_s = scaler.transform(Xtr)
Xte_s = scaler.transform(Xte)

scaled = KNeighborsClassifier(n_neighbors=5).fit(Xtr_s, ytr)
print(f"held-out accuracy, scaled features: "
      f"{accuracy_score(yte, scaled.predict(Xte_s)):.3f}")

In [ ]:
print("   k    train acc   test acc")
for k in (1, 5, 15, 61, 101, 163):
    m = KNeighborsClassifier(n_neighbors=k).fit(Xtr_s, ytr)
    tr_acc = accuracy_score(ytr, m.predict(Xtr_s))
    te_acc = accuracy_score(yte, m.predict(Xte_s))
    print(f"  {k:3}     {tr_acc:.3f}      {te_acc:.3f}")

In [ ]:
scaler = StandardScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr)
Xte_s = scaler.transform(Xte)

model = KNeighborsClassifier(n_neighbors=5).fit(Xtr_s, ytr)

run_tests([
    ("scaled held-out accuracy", round(
        accuracy_score(yte, model.predict(Xte_s)), 3), 0.964),
    ("scaler learned from train only", round(
        float(scaler.mean_[0]), 1), 42.1),
])

In [ ]:
import math

def euclidean(p, q):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(p, q)))

def knn_predict(train_X, train_y, probe, k):
    order = sorted(range(len(train_X)),
                   key=lambda i: euclidean(train_X[i], probe))
    votes = [train_y[i] for i in order[:k]]
    return 1 if sum(votes) > k / 2 else 0

pts = [(0.0, 0.0), (1.0, 0.0), (0.0, 1.0), (5.0, 5.0), (6.0, 5.0)]
lbl = [0, 0, 0, 1, 1]

run_tests([
    ("3-4-5 triangle", euclidean((0, 0), (3, 4)), 5.0),
    ("distance to itself", euclidean((2.5, 7.0), (2.5, 7.0)), 0.0),
    ("deep in the 0-cluster", knn_predict(pts, lbl, (0.5, 0.5), 3), 0),
    ("deep in the 1-cluster", knn_predict(pts, lbl, (5.5, 5.0), 3), 1),
    ("midfield, ask everyone", knn_predict(pts, lbl, (3.0, 3.0), 5), 0),
])